# Gemini Pro online grader 初始化

这个辅助 Notebook 安装官方 Antigravity CLI (`agy`)，引导你用 Google AI Pro 账号完成 OAuth，并验证结构化在线判分。它不会进入 Step 00–08 的合并实验文件，也不会保存 OAuth token。

建议顺序：先运行 Step 00 写出实验配置，再运行这里，最后回到 Step 02。先用 10 条 pilot 验证额度和速度，再扩大到完整数据。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/ZIP-RC-Colab")]
REPO = next(
    (path for path in candidates if (path / "src" / "evaluate_and_label_rollouts.py").exists()),
    None,
)
if REPO is None:
    raise FileNotFoundError("找不到 ZIP-RC 仓库。")

os.environ["PATH"] = f"{Path.home() / '.local/bin'}{os.pathsep}{os.environ['PATH']}"
AGY = Path.home() / ".local/bin/agy"

import pandas as pd
from IPython.display import display

print("Repository:", REPO)
print("Expected agy path:", AGY)

## 1. 安装官方 Antigravity CLI

此 cell 从 Google 官方地址下载安装脚本到 `/tmp`，然后安装到当前 runtime 的 `~/.local/bin/agy`。runtime 被删除后通常需要重新安装。

In [ ]:
INSTALL_OR_UPGRADE_AGY = True  # @param {type:"boolean"}
installer = Path("/tmp/antigravity-install.sh")
if INSTALL_OR_UPGRADE_AGY:
    if shutil.which("curl") is None:
        raise FileNotFoundError("runtime 中找不到 curl。")
    subprocess.run(
        ["curl", "-fsSL", "https://antigravity.google/cli/install.sh", "-o", str(installer)],
        check=True,
    )
    subprocess.run(["bash", str(installer), "--skip-aliases"], check=True)

if not AGY.exists():
    raise FileNotFoundError(f"安装结束后仍未找到 {AGY}")
subprocess.run([str(AGY), "--version"], check=True)

## 2. 用 Google AI Pro 账号登录

OAuth 必须由你亲自确认。**推荐在 Colab Enterprise 当前 runtime 的 Terminal 中运行：**

```bash
~/.local/bin/agy
```

若远端无法打开浏览器，CLI 会显示授权 URL：在本机浏览器打开它，用购买 Google AI Pro 的账号登录，再把授权码粘回 Terminal。登录成功后可退出交互界面并回到这里。

不要把 `~/.gemini/`、授权码或 token 复制到仓库、Google Drive、Notebook 输出或 artifacts。

In [ ]:
TRY_INTERACTIVE_LOGIN_IN_CELL = False  # @param {type:"boolean"}
print("请在 Colab Enterprise Terminal 运行：")
print(f"  {AGY}")
if TRY_INTERACTIVE_LOGIN_IN_CELL:
    print("正在当前 cell 启动交互登录；若前端不能输入，请中断并改用 Terminal。")
    subprocess.run([str(AGY)], check=False)
else:
    print("完成 OAuth 后，再运行下面的模型列表 cell。")

## 3. 选择账号实际可用的 Gemini Pro slug

`gemini-pro-online` 是本项目里的别名；真正传给 `agy --model` 的 slug 以 `agy models` 当前输出为准。下面默认选择列表中的第一个 Gemini Pro。

In [ ]:
models_result = subprocess.run(
    [str(AGY), "models"],
    capture_output=True,
    text=True,
)
if models_result.returncode != 0:
    raise RuntimeError(
        "agy models 失败；请先在 Terminal 完成 OAuth。\n"
        + (models_result.stderr.strip() or models_result.stdout.strip())
    )
print(models_result.stdout)
model_rows = [line.split(maxsplit=1) for line in models_result.stdout.splitlines() if line.strip()]
pro_models = [
    fields[0]
    for fields in model_rows
    if fields[0].startswith("gemini-") and "pro" in fields[0].lower()
]
if not pro_models:
    raise ValueError("当前账号的 `agy models` 没有返回 Gemini Pro slug。")
MODEL_SLUG_OVERRIDE = ""  # @param {type:"string"}
MODEL_SLUG = MODEL_SLUG_OVERRIDE.strip() or pro_models[0]
if MODEL_SLUG not in pro_models:
    raise ValueError(f"{MODEL_SLUG!r} 不在当前账号返回的 Gemini Pro 列表中。")
display(pd.DataFrame({"Gemini Pro slug": pro_models, "selected": [m == MODEL_SLUG for m in pro_models]}))
print("Selected:", MODEL_SLUG)

## 4. 结构化 smoke test

成功标准是 `status == SUCCESS` 且 `structured_output.correct is True`。这一步同时验证 OAuth、模型权限和 JSON Schema；不成功时不要直接跑数百条 grader。

In [ ]:
schema = json.dumps(
    {
        "type": "object",
        "properties": {"correct": {"type": "boolean"}},
        "required": ["correct"],
        "additionalProperties": False,
    },
    separators=(",", ":"),
)
smoke = subprocess.run(
    [
        str(AGY),
        "-p",
        "Determine whether 2+2 is equivalent to 4. Return correct=true. Do not use tools.",
        "--model",
        MODEL_SLUG,
        "--effort",
        "low",
        "--output-format",
        "json",
        "--json-schema",
        schema,
        "--print-timeout",
        "5m",
        "--sandbox",
    ],
    capture_output=True,
    cwd="/tmp",
    text=True,
    timeout=360,
)
if smoke.returncode != 0:
    raise RuntimeError(smoke.stderr.strip() or smoke.stdout.strip())
envelope = json.loads(smoke.stdout)
checks = pd.DataFrame(
    [
        {"check": "agy installed", "passed": AGY.exists(), "detail": str(AGY)},
        {"check": "authenticated request", "passed": envelope.get("status") == "SUCCESS", "detail": envelope.get("status")},
        {"check": "structured boolean", "passed": envelope.get("structured_output", {}).get("correct") is True, "detail": envelope.get("structured_output")},
    ]
)
display(checks)
if not checks["passed"].all():
    raise RuntimeError(f"Gemini Pro smoke test 未通过：{envelope}")
print("✅ Gemini Pro online grader 可用。")

## 5. 启用在线 grader

这里只保存非敏感的模型 slug，并把已存在的 Step 00 实验配置切换到 `gemini-pro-online`。如果实验配置不存在，请先运行 Step 00，再重跑此 cell。

In [ ]:
ENABLE_FOR_EXPERIMENT = True  # @param {type:"boolean"}
artifacts = REPO / "artifacts"
artifacts.mkdir(parents=True, exist_ok=True)
online_config_path = artifacts / "antigravity_config.json"
online_config_path.write_text(
    json.dumps({"backend": "gemini-pro-online", "model": MODEL_SLUG}, indent=2) + "\n",
    encoding="utf-8",
)

experiment_config_path = artifacts / "experiment_config.json"
if ENABLE_FOR_EXPERIMENT:
    if not experiment_config_path.exists():
        raise FileNotFoundError("请先运行 Step 00，再重跑此 cell。")
    experiment_config = json.loads(experiment_config_path.read_text(encoding="utf-8"))
    experiment_config["grader_model_id"] = "gemini-pro-online"
    experiment_config_path.write_text(
        json.dumps(experiment_config, indent=2) + "\n",
        encoding="utf-8",
    )

print("Saved non-secret config:", online_config_path)
print("grader_model_id: gemini-pro-online")
print("下一步：回到 Step 02；建议先让 pilot 只包含 10 条 finished rollout。")